# HSC S23b Training Sample QA

Publication-quality assessment of the HSC S23b deep training catalog.

- **Source catalog**: `hsc_s23b_deep_matched_train_SFR_v3.fits` (453,652 objects)
- **Spec-z sources**: COSMOSWeb2025_v1 (~310k) + DESI_DR1 (~143k) + Dual (554)
- **Bands**: grizy (5-band HSC)
- **Flux units**: nJy (AB ZP = 31.4)
- **Output**: Cleaned PhotoData HDF5 files per flux type, ready for `run_pipeline()`
- **Schema**: See `docs/catalog_schema.md` for full 343-column documentation

## 0. Configuration

In [ ]:
from pathlib import Path

# --- Paths ---
CATALOG_PATH = Path(
    "/Users/shuang/work/hsc/photoz/s23b_photoz_calib_v3/"
    "hsc_s23b_deep_matched_train_SFR_v3.fits"
)
OUTPUT_DIR = Path("output")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# --- Bands ---
BANDS = ["g", "r", "i", "z", "y"]
N_BANDS = len(BANDS)

# --- Full flux type registry (all 8 types) ---
FLUX_TYPE_REGISTRY = {
    "cmodel":      ("{b}_cmodel_flux",                         "{b}_cmodel_fluxerr"),
    "cmodel_exp":  ("{b}_cmodel_exp_flux",                     "{b}_cmodel_exp_fluxerr"),
    "cmodel_dev":  ("{b}_cmodel_dev_flux",                     "{b}_cmodel_dev_fluxerr"),
    "psf":         ("{b}_psfflux_flux",                        "{b}_psfflux_fluxerr"),
    "gaap_optimal":("{b}_gaapflux_1_15x_optimal_flux",         "{b}_gaapflux_1_15x_optimal_fluxerr"),
    "gaap_psf":    ("{b}_gaapflux_1_15x_psfflux_flux",         "{b}_gaapflux_1_15x_psfflux_fluxerr"),
    "convolved":   ("{b}_convolvedflux_3_15_flux",             "{b}_convolvedflux_3_15_fluxerr"),
    "undeblended": ("{b}_undeblended_convolvedflux_3_15_flux", "{b}_undeblended_convolvedflux_3_15_fluxerr"),
}

# Types for detailed per-type analysis (sections 5+)
SELECTED_FLUX_TYPES = ["cmodel", "gaap_optimal", "convolved"]

# Friendly display labels
PHOT_LABELS = {
    "cmodel": "CModel", "cmodel_exp": "CModel-Exp", "cmodel_dev": "CModel-Dev",
    "psf": "PSF", "gaap_optimal": "GAaP", "gaap_psf": "GAaP-PSF",
    "convolved": "Conv", "undeblended": "Undeblend",
}

# --- Photometry ---
AB_ZP = 31.4

# --- Quality cuts ---
SNR_THRESHOLD = 3.0
Z_MIN = 0.0
Z_MAX = 7.0
ALLOWED_OBJECT_TYPES = {"G", "Q", "G/G", "G/Q", "Q/G", "Q/Q"}

# --- KDE bandwidth ---
KDE_BANDWIDTH_FRACTION = 0.01
KDE_BANDWIDTH_FLOOR = 0.01

# --- Plotting ---
DPI = 150

## 1. Imports + Helpers

In [ ]:
import numpy as np
import matplotlib
matplotlib.rcParams["figure.dpi"] = DPI
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from astropy.table import Table

import sys, yaml
sys.path.insert(0, str(Path("..").resolve()))
from frankenz.io import PhotoData, write_hdf5, read_hdf5

from s23b_plot import (
    plot_scale_dz, plot_mag_dz, plot_mag_z, plot_color_z,
    plot_color_color, plot_z_hist_by_source, plot_z_compare,
    plot_completeness_heatmap, SOURCE_STYLE,
)

In [ ]:
def flux_to_ab_mag(flux_njy):
    """Convert nJy flux to AB magnitude. Returns NaN for non-positive flux."""
    with np.errstate(divide="ignore", invalid="ignore"):
        mag = -2.5 * np.log10(flux_njy) + AB_ZP
    return mag


def compute_snr(flux, flux_err):
    """Signal-to-noise ratio. Returns NaN where flux_err <= 0."""
    with np.errstate(divide="ignore", invalid="ignore"):
        snr = flux / flux_err
    snr[flux_err <= 0] = np.nan
    return snr


def extract_photometry(table, flux_type, bands=BANDS):
    """Extract (flux, flux_err) arrays for a given flux type."""
    flux_pat, err_pat = FLUX_TYPE_REGISTRY[flux_type]
    flux_cols = [flux_pat.format(b=b) for b in bands]
    err_cols = [err_pat.format(b=b) for b in bands]
    flux = np.column_stack([np.array(table[c], dtype=np.float64) for c in flux_cols])
    flux_err = np.column_stack([np.array(table[c], dtype=np.float64) for c in err_cols])
    return flux, flux_err


def extract_extinction(table, bands=BANDS):
    """Extract per-band galactic extinction A_b array."""
    return np.column_stack(
        [np.array(table[f"a_{b}"], dtype=np.float64) for b in bands]
    )


def apply_extinction_correction(flux_njy, extinction_ab):
    """Correct for galactic extinction. Errors are NOT corrected."""
    raw_mag = flux_to_ab_mag(flux_njy)
    corrected_mag = raw_mag - extinction_ab
    corrected_flux = 10.0 ** (-0.4 * (corrected_mag - AB_ZP))
    return corrected_flux


def completeness_summary(flux, bands=BANDS):
    """Compute per-band and all-band completeness (finite + positive)."""
    valid = np.isfinite(flux) & (flux > 0)
    n = len(flux)
    per_band = {b: valid[:, i].sum() / n for i, b in enumerate(bands)}
    all_bands = valid.all(axis=1).sum() / n
    return per_band, all_bands


def compute_colors(mag):
    """Compute adjacent-band colors from magnitude array."""
    band_idx = {b: i for i, b in enumerate(BANDS)}
    colors = {}
    for b1, b2 in [("g", "r"), ("r", "i"), ("i", "z"), ("z", "y")]:
        colors[f"{b1}-{b2}"] = mag[:, band_idx[b1]] - mag[:, band_idx[b2]]
    return colors

## 2. Data Loading

In [ ]:
print(f"Loading {CATALOG_PATH.name} ...")
cat = Table.read(CATALOG_PATH)
print(f"  {len(cat):,} rows, {len(cat.colnames)} columns")

In [ ]:
# Extract shared arrays
redshift = np.array(cat["redshift"], dtype=np.float64)
redshift_err = np.array(cat["redshift_err"], dtype=np.float64)
specz_sources = np.array(cat["specz_sources"], dtype=str)
object_type = np.array(cat["object_type"], dtype=str)
sample_crossval = np.array(cat["sample_crossval"], dtype=int)
ra = np.array(cat["ra"], dtype=np.float64)
dec = np.array(cat["dec"], dtype=np.float64)

# Source masks (reused throughout)
mask_desi = specz_sources == "DESI_DR1"
mask_cosmos = specz_sources == "COSMOSWeb2025_v1"
mask_dual = specz_sources == "DESI_DR1/COSMOSWeb2025_v1"

print(f"Redshift range: [{redshift.min():.4f}, {redshift.max():.4f}]")
print(f"Median z: {np.median(redshift):.4f}")
print(f"Sources: DESI={mask_desi.sum():,}, COSMOSWeb={mask_cosmos.sum():,}, Dual={mask_dual.sum():,}")

## 3. Redshift QA (Flux-Independent)

In [ ]:
# --- Figure 1: Redshift distribution by source (linear + log scale) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

plot_z_hist_by_source(redshift, specz_sources, ax=ax1)
ax1.set_title("Redshift Distribution by Source")

plot_z_hist_by_source(redshift, specz_sources, ax=ax2)
ax2.set_yscale("log")
ax2.set_title("Redshift Distribution (log scale)")
ax2.set_ylabel(r"$\rm Count\ (log)$", fontsize=20)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "redshift_distribution_by_source.png", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
# --- Redshift error quality by source ---
print("Redshift error analysis by source:")
print(f"{'Source':<35} {'N':>8} {'zerr<0':>8} {'zerr=0':>8} {'0<zerr<1':>10} {'zerr>=1':>8}")
print("-" * 80)
for src in sorted(set(specz_sources)):
    mask = specz_sources == src
    ze = redshift_err[mask]
    n = mask.sum()
    n_neg = (ze < 0).sum()
    n_zero = (ze == 0).sum()
    n_good = ((ze > 0) & (ze < 1)).sum()
    n_large = (ze >= 1).sum()
    print(f"{src:<35} {n:>8,} {n_neg:>8,} {n_zero:>8,} {n_good:>10,} {n_large:>8,}")
    print(f"{'':35} {'':>8} {n_neg/n*100:>7.1f}% {n_zero/n*100:>7.1f}% {n_good/n*100:>9.1f}% {n_large/n*100:>7.1f}%")

In [ ]:
# --- Figure 2: Scale factor vs log10(dz/(1+z)) --- COSMOSWeb + DESI z distribution ---
# DESI has sentinel zerr=-1, so only COSMOSWeb has usable errors
cosmos_good = mask_cosmos & (redshift_err > 0) & (redshift_err < 1)
scale_cosmos = 1.0 / (1.0 + redshift[cosmos_good])
logdz_cosmos = np.log10(redshift_err[cosmos_good] / (1.0 + redshift[cosmos_good]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

plot_scale_dz(scale_cosmos, logdz_cosmos, ax=ax1,
              cmap="inferno", color="orangered",
              label=r"$\rm COSMOSWeb$")

# DESI: show z distribution (no zerr available)
ax2.hist(redshift[mask_desi], bins=np.arange(0, 7, 0.03),
         color="steelblue", alpha=0.7, edgecolor="white", linewidth=0.3)
ax2.set_xlabel(r"$\rm Redshift$", fontsize=25)
ax2.set_ylabel(r"$\rm Count$", fontsize=25)
ax2.text(0.60, 0.9, r"$\rm DESI$", fontsize=30, transform=ax2.transAxes)
ax2.text(0.60, 0.80, r"$\rm (no\ z_{err})$", fontsize=18,
         transform=ax2.transAxes, color="gray")
ax2.grid(linestyle="--", linewidth=2, alpha=0.6)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "scale_factor_vs_logdz_by_source.png", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 3: Spec-z source overlap analysis ---
print(f"Spec-z source overlap:")
print(f"  DESI-only:      {mask_desi.sum():>8,}")
print(f"  COSMOSWeb-only: {mask_cosmos.sum():>8,}")
print(f"  Dual (both):    {mask_dual.sum():>8,}")
print(f"  Total:          {len(cat):>8,}")

if mask_dual.sum() > 10:
    z_dual = redshift[mask_dual]
    ze_dual = redshift_err[mask_dual]
    print(f"\nDual object statistics:")
    print(f"  z median: {np.median(z_dual):.4f}")
    print(f"  z range:  [{z_dual.min():.4f}, {z_dual.max():.4f}]")
    print(f"  zerr median: {np.median(ze_dual):.6f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

    # Dual z vs full sample
    z_bins = np.arange(0, 5, 0.05)
    ax1.hist(redshift, bins=z_bins, alpha=0.3, color="gray",
             label=f"All (N={len(redshift):,})", density=True)
    ax1.hist(z_dual, bins=z_bins, alpha=0.7, color="forestgreen",
             label=f"Dual (N={mask_dual.sum():,})", density=True)
    ax1.set_xlabel(r"$\rm Redshift$", fontsize=20)
    ax1.set_ylabel(r"$\rm Density$", fontsize=20)
    ax1.set_title("Dual-Source Objects vs Full Sample")
    ax1.legend(fontsize=12)
    ax1.grid(linestyle="--", linewidth=2, alpha=0.6)

    # Dual zerr distribution
    ax2.hist(ze_dual[ze_dual > 0], bins=50, color="forestgreen", alpha=0.7)
    ax2.set_xlabel(r"$\rm Redshift\ Error$", fontsize=20)
    ax2.set_ylabel(r"$\rm Count$", fontsize=20)
    ax2.set_title(f"Dual-Source zerr (median={np.median(ze_dual[ze_dual > 0]):.5f})")
    ax2.grid(linestyle="--", linewidth=2, alpha=0.6)

    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "specz_source_overlap.png", dpi=DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# --- Figure 4: Object type distribution ---
types_unique, type_counts = np.unique(object_type, return_counts=True)
sort_idx = np.argsort(-type_counts)
types_unique = types_unique[sort_idx]
type_counts = type_counts[sort_idx]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(len(types_unique)), type_counts, color="steelblue", edgecolor="white")
ax.set_xticks(range(len(types_unique)))
ax.set_xticklabels(types_unique)
ax.set_ylabel("Count")
ax.set_title("Object Type Distribution")
for bar, count in zip(bars, type_counts):
    pct = count / len(cat) * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{count:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "object_type_distribution.png", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 5: Cross-validation fold balance ---
folds_unique, fold_counts = np.unique(sample_crossval, return_counts=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(folds_unique, fold_counts, color="steelblue", edgecolor="white")
ax.set_xlabel("Cross-validation Fold")
ax.set_ylabel("Count")
ax.set_title(f"Cross-validation Fold Balance ({len(folds_unique)} folds)")
for bar, count in zip(bars, fold_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{count:,}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(folds_unique)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "crossval_fold_balance.png", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 6: Spatial distribution ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# By source
for m, src, c in [(mask_cosmos, "COSMOSWeb", "orangered"),
                  (mask_desi, "DESI", "steelblue"),
                  (mask_dual, "Dual", "forestgreen")]:
    ax1.scatter(ra[m], dec[m], s=0.1, alpha=0.3, color=c,
               label=src, rasterized=True)
ax1.set_xlabel("RA [deg]")
ax1.set_ylabel("Dec [deg]")
ax1.set_title("Spatial Distribution by Source")
ax1.legend(markerscale=20, fontsize=10)
ax1.invert_xaxis()

# Density
h = ax2.hist2d(ra, dec, bins=[200, 200], cmin=1, cmap="viridis", norm=LogNorm())
plt.colorbar(h[3], ax=ax2, label="Count")
ax2.set_xlabel("RA [deg]")
ax2.set_ylabel("Dec [deg]")
ax2.set_title("Spatial Density")
ax2.invert_xaxis()

fig.tight_layout()
fig.savefig(FIGURE_DIR / "spatial_distribution.png", dpi=DPI, bbox_inches="tight")
plt.show()

## 4. Completeness Overview (All 8 Flux Types)

In [ ]:
# Validate all registry columns exist
all_flux_types = list(FLUX_TYPE_REGISTRY.keys())
for ft in all_flux_types:
    flux_pat, err_pat = FLUX_TYPE_REGISTRY[ft]
    for b in BANDS:
        assert flux_pat.format(b=b) in cat.colnames, f"Missing: {flux_pat.format(b=b)}"
        assert err_pat.format(b=b) in cat.colnames, f"Missing: {err_pat.format(b=b)}"
print(f"All {len(all_flux_types)} flux types validated.")

In [ ]:
# --- Figure 7: Completeness heatmap (all 8 flux types) ---
completeness_matrix = []
ft_labels = []

print(f"{'Flux Type':<15} {'g':>7} {'r':>7} {'i':>7} {'z':>7} {'y':>7} {'All-band':>10}")
print("-" * 65)

for ft in all_flux_types:
    flux, _ = extract_photometry(cat, ft)
    per_band, all_bands = completeness_summary(flux)
    row = [per_band[b] * 100 for b in BANDS]
    completeness_matrix.append(row)
    ft_labels.append(ft)
    print(f"{ft:<15} {row[0]:>6.1f}% {row[1]:>6.1f}% {row[2]:>6.1f}% "
          f"{row[3]:>6.1f}% {row[4]:>6.1f}% {all_bands*100:>9.1f}%")

completeness_matrix = np.array(completeness_matrix)

fig = plt.figure(figsize=(10, 5))
ax = plt.subplot(111)
plot_completeness_heatmap(completeness_matrix, ft_labels, BANDS, ax=ax)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "completeness_heatmap_all_types.png", dpi=DPI, bbox_inches="tight")
plt.show()

## 5. Per-Flux-Type Photometry QA

In [ ]:
# Extract photometry for selected types
photometry = {}
for ft in SELECTED_FLUX_TYPES:
    flux, flux_err = extract_photometry(cat, ft)
    photometry[ft] = (flux, flux_err)
    print(f"  {ft}: flux shape={flux.shape}, "
          f"finite={np.isfinite(flux).all(axis=1).mean()*100:.1f}% all-band")

extinction = extract_extinction(cat)
print(f"\nExtinction: median A_b per band = "
      f"{[f'{np.nanmedian(extinction[:, i]):.4f}' for i, b in enumerate(BANDS)]}")

In [ ]:
# --- Figures 8-10: Mag-z + color-z panels (DESI vs COSMOSWeb), per flux type ---
for ft in SELECTED_FLUX_TYPES:
    flux, _ = photometry[ft]
    mag = flux_to_ab_mag(flux)
    i_mag = mag[:, 2]
    colors = compute_colors(mag)
    phot_label = PHOT_LABELS.get(ft, ft)

    fig, axes = plt.subplots(3, 2, figsize=(13, 19))

    for col_idx, (m, src_key, cm) in enumerate([
        (mask_desi, "DESI_DR1", "viridis"),
        (mask_cosmos, "COSMOSWeb2025_v1", "inferno"),
    ]):
        style = SOURCE_STYLE[src_key]
        valid = m & np.isfinite(i_mag)

        # Row 1: i-mag vs z
        plot_mag_z(
            i_mag[valid], redshift[valid], ax=axes[0, col_idx],
            cmap=cm, color=style["color"], label=style["label"],
            phot=phot_label, cmin=3, add_colorbar=False,
        )

        # Row 2: g-r vs z
        gr = colors["g-r"][valid]
        gr_ok = np.isfinite(gr)
        plot_color_z(
            gr[gr_ok], redshift[valid][gr_ok], ax=axes[1, col_idx],
            cmap=cm, scatter_color=style["color"],
            band_1="g", band_2="r", xlim=(-0.99, 2.99),
            phot=phot_label, cmin=3, add_colorbar=False,
        )

        # Row 3: i-z vs z
        iz = colors["i-z"][valid]
        iz_ok = np.isfinite(iz)
        plot_color_z(
            iz[iz_ok], redshift[valid][iz_ok], ax=axes[2, col_idx],
            cmap=cm, scatter_color=style["color"],
            band_1="i", band_2="z", xlim=(-0.99, 2.49),
            phot=phot_label, cmin=3, add_colorbar=False,
        )

    fig.suptitle(f"Magnitude/Color vs Redshift: {phot_label}", fontsize=16, y=1.01)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"mag_color_z_{ft}.png", dpi=DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# --- Figures 11-13: Redshift error relations per flux type (COSMOSWeb only) ---
for ft in SELECTED_FLUX_TYPES:
    flux, _ = photometry[ft]
    mag = flux_to_ab_mag(flux)
    i_mag = mag[:, 2]
    colors = compute_colors(mag)
    phot_label = PHOT_LABELS.get(ft, ft)

    # COSMOSWeb with valid zerr
    valid_zerr = mask_cosmos & (redshift_err > 0) & (redshift_err < 1) & np.isfinite(i_mag)
    scale = 1.0 / (1.0 + redshift[valid_zerr])
    logdz = np.log10(redshift_err[valid_zerr] / (1.0 + redshift[valid_zerr]))
    i_m = i_mag[valid_zerr]
    gr = colors["g-r"][valid_zerr]
    iz = colors["i-z"][valid_zerr]

    fig, axes = plt.subplots(2, 2, figsize=(13, 13))

    # Top-left: scale factor vs log10(dz/(1+z))
    plot_scale_dz(
        scale, logdz, ax=axes[0, 0],
        cmap="inferno", color="orangered",
        label=r"$\rm COSMOSWeb$", add_colorbar=False,
    )

    # Top-right: i-mag vs log10(dz/(1+z))
    plot_mag_dz(
        i_m, logdz, ax=axes[0, 1],
        cmap="inferno", color="orangered",
        phot=phot_label, add_colorbar=False,
    )

    # Bottom-left: g-r color vs log10(dz/(1+z))
    gr_ok = np.isfinite(gr)
    axes[1, 0].scatter(gr[gr_ok], logdz[gr_ok], s=2, alpha=0.2, color="orangered")
    axes[1, 0].hist2d(
        gr[gr_ok], logdz[gr_ok], bins=[120, 100], cmin=2, cmap="inferno",
        range=[[-1, 3], [-5.9, 0.9]], norm=LogNorm(),
    )
    axes[1, 0].set_xlim(-1, 3)
    axes[1, 0].set_ylim(-5.9, 0.9)
    axes[1, 0].set_xlabel(f"{phot_label} g-r [mag]", fontsize=20)
    axes[1, 0].set_ylabel(r"$\log_{10}[\delta z / (1 + z)]$", fontsize=20)
    axes[1, 0].grid(linestyle="--", linewidth=2, alpha=0.6)

    # Bottom-right: color-color (g-r vs i-z)
    iz_ok = np.isfinite(iz)
    both_ok = gr_ok & iz_ok
    plot_color_color(
        gr[both_ok], iz[both_ok], ax=axes[1, 1],
        cmap="inferno", scatter_color="orangered",
        xlabel=f"{phot_label} g-r [mag]",
        ylabel=f"{phot_label} i-z [mag]",
        xlim=(-1, 3), ylim=(-1, 2),
        add_colorbar=False,
    )

    fig.suptitle(f"Redshift Error Relations: {phot_label} (COSMOSWeb)",
                 fontsize=16, y=1.01)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"zerr_relations_{ft}.png", dpi=DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# --- Figures 14-16: SNR distribution per flux type ---
for ft in SELECTED_FLUX_TYPES:
    flux, flux_err = photometry[ft]
    snr = compute_snr(flux, flux_err)

    fig, axes = plt.subplots(1, N_BANDS, figsize=(15, 3.5), sharey=True)
    for i, (ax, b) in enumerate(zip(axes, BANDS)):
        valid = np.isfinite(snr[:, i]) & (snr[:, i] > 0)
        ax.hist(np.log10(snr[valid, i]), bins=60, color="steelblue",
                alpha=0.7, edgecolor="white", linewidth=0.3)
        med = np.nanmedian(snr[:, i])
        ax.axvline(np.log10(med), color="red", ls="--", lw=1.2)
        ax.set_xlabel("log10(SNR)")
        ax.set_title(f"{b}-band (med={med:.1f})")
    axes[0].set_ylabel("Count")
    fig.suptitle(f"SNR Distribution: {ft}", fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"snr_distribution_{ft}.png", dpi=DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# --- Figures 17-19: Magnitude distribution per flux type (source-split) ---
for ft in SELECTED_FLUX_TYPES:
    flux, _ = photometry[ft]
    mag = flux_to_ab_mag(flux)

    fig, axes = plt.subplots(1, N_BANDS, figsize=(15, 3.5), sharey=True)
    for i, (ax, b) in enumerate(zip(axes, BANDS)):
        valid = np.isfinite(mag[:, i])
        ax.hist(mag[valid & mask_desi, i], bins=80, range=(18, 30),
                color="steelblue", alpha=0.5, label="DESI")
        ax.hist(mag[valid & mask_cosmos, i], bins=80, range=(18, 30),
                color="orangered", alpha=0.5, label="COSMOSWeb")
        med = np.nanmedian(mag[:, i])
        ax.axvline(med, color="k", ls="--", lw=1.2)
        ax.set_xlabel("AB mag")
        ax.set_title(f"{b}-band (med={med:.1f})")
    axes[0].set_ylabel("Count")
    axes[-1].legend(fontsize=8)
    fig.suptitle(f"Magnitude Distribution: {ft}", fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"mag_distribution_{ft}.png", dpi=DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# --- Figures 20-22: Redshift in i-mag bins (source-split) ---
for ft in SELECTED_FLUX_TYPES:
    flux, _ = photometry[ft]
    i_mag = flux_to_ab_mag(flux[:, 2])

    mag_edges = [20, 22, 23, 24, 25, 26, 28]
    n_mag_bins = len(mag_edges) - 1

    fig, axes = plt.subplots(1, n_mag_bins, figsize=(3 * n_mag_bins, 3.5), sharey=True)
    z_bins = np.arange(0, 5, 0.05)
    for k in range(n_mag_bins):
        ax = axes[k]
        in_bin = (i_mag >= mag_edges[k]) & (i_mag < mag_edges[k + 1]) & np.isfinite(i_mag)
        ax.hist(redshift[in_bin & mask_desi], bins=z_bins, alpha=0.5,
                color="steelblue", label="DESI")
        ax.hist(redshift[in_bin & mask_cosmos], bins=z_bins, alpha=0.5,
                color="orangered", label="COSMOSWeb")
        ax.set_xlabel("Redshift")
        ax.set_title(f"i=[{mag_edges[k]},{mag_edges[k+1]})\nN={in_bin.sum():,}", fontsize=9)
    axes[0].set_ylabel("Count")
    axes[-1].legend(fontsize=7)
    fig.suptitle(f"Redshift by i-mag Bin: {ft}", fontsize=13, y=1.05)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"z_in_mag_bins_{ft}.png", dpi=DPI, bbox_inches="tight")
    plt.show()

## 6. Cross-Flux-Type Comparison

In [ ]:
# --- Figure 23: Cross-flux i-mag + (g-r) color comparison (2D histogram) ---
if len(SELECTED_FLUX_TYPES) >= 2:
    ft_a, ft_b = SELECTED_FLUX_TYPES[0], SELECTED_FLUX_TYPES[1]
    flux_a, _ = photometry[ft_a]
    flux_b, _ = photometry[ft_b]
    mag_a = flux_to_ab_mag(flux_a[:, 2])
    mag_b = flux_to_ab_mag(flux_b[:, 2])
    valid = np.isfinite(mag_a) & np.isfinite(mag_b)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

    # i-band magnitude
    lim = (18, 28)
    ax1.scatter(mag_a[valid], mag_b[valid], s=2, alpha=0.2, color="steelblue")
    ax1.hist2d(mag_a[valid], mag_b[valid], bins=[120, 120], cmin=2,
               cmap="viridis", range=[list(lim), list(lim)], norm=LogNorm())
    ax1.plot(lim, lim, "r--", lw=2)
    ax1.set_xlabel(f"{ft_a} i-mag", fontsize=15)
    ax1.set_ylabel(f"{ft_b} i-mag", fontsize=15)
    ax1.set_title(f"i-band: {ft_a} vs {ft_b}")
    ax1.set_xlim(lim); ax1.set_ylim(lim)
    ax1.grid(linestyle="--", linewidth=2, alpha=0.6)

    diff = mag_b[valid] - mag_a[valid]
    med_diff = np.median(diff)
    mad_diff = 1.4826 * np.median(np.abs(diff - med_diff))
    ax1.text(0.03, 0.97, f"med offset = {med_diff:.4f}\nMAD = {mad_diff:.4f}",
             transform=ax1.transAxes, va="top", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

    # (g-r) color
    mag_full_a = flux_to_ab_mag(flux_a)
    mag_full_b = flux_to_ab_mag(flux_b)
    gr_a = mag_full_a[:, 0] - mag_full_a[:, 1]
    gr_b = mag_full_b[:, 0] - mag_full_b[:, 1]
    valid_c = np.isfinite(gr_a) & np.isfinite(gr_b)

    clim = (-1, 3)
    ax2.scatter(gr_a[valid_c], gr_b[valid_c], s=2, alpha=0.2, color="steelblue")
    ax2.hist2d(gr_a[valid_c], gr_b[valid_c], bins=[120, 120], cmin=2,
               cmap="viridis", range=[list(clim), list(clim)], norm=LogNorm())
    ax2.plot(clim, clim, "r--", lw=2)
    ax2.set_xlabel(f"{ft_a} (g-r)", fontsize=15)
    ax2.set_ylabel(f"{ft_b} (g-r)", fontsize=15)
    ax2.set_title(f"(g-r) color: {ft_a} vs {ft_b}")
    ax2.set_xlim(clim); ax2.set_ylim(clim)
    ax2.grid(linestyle="--", linewidth=2, alpha=0.6)

    cdiff = gr_b[valid_c] - gr_a[valid_c]
    ax2.text(0.03, 0.97,
             f"med offset = {np.median(cdiff):.4f}\n"
             f"MAD = {1.4826 * np.median(np.abs(cdiff - np.median(cdiff))):.4f}",
             transform=ax2.transAxes, va="top", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

    fig.suptitle(f"Cross-Flux Comparison: {ft_a} vs {ft_b}", fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"cross_flux_comparison_{ft_a}_vs_{ft_b}.png",
                dpi=DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# --- Figure 24: Flux ratio analysis per band ---
if len(SELECTED_FLUX_TYPES) >= 2:
    ft_a, ft_b = SELECTED_FLUX_TYPES[0], SELECTED_FLUX_TYPES[1]
    flux_a, _ = photometry[ft_a]
    flux_b, _ = photometry[ft_b]

    fig, axes = plt.subplots(1, N_BANDS, figsize=(15, 3.5), sharey=True)
    for i, (ax, b) in enumerate(zip(axes, BANDS)):
        ok = ((flux_a[:, i] > 0) & (flux_b[:, i] > 0)
              & np.isfinite(flux_a[:, i]) & np.isfinite(flux_b[:, i]))
        log_ratio = np.log10(flux_b[ok, i] / flux_a[ok, i])
        log_ratio = log_ratio[(log_ratio > -1) & (log_ratio < 1)]
        ax.hist(log_ratio, bins=100, color="steelblue", alpha=0.7,
                edgecolor="white", linewidth=0.3)
        med = np.median(log_ratio)
        ax.axvline(med, color="red", ls="--", lw=1.5)
        ax.axvline(0, color="k", ls=":", lw=1)
        ax.set_xlabel(f"log10({ft_b}/{ft_a})")
        ax.set_title(f"{b}-band (med={med:.4f})")
    axes[0].set_ylabel("Count")
    fig.suptitle(f"Flux Ratio: {ft_b} / {ft_a}", fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"flux_ratio_{ft_a}_vs_{ft_b}.png",
                dpi=DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# --- Figure 25: Magnitude-dependent residual ---
if len(SELECTED_FLUX_TYPES) >= 2:
    ft_a, ft_b = SELECTED_FLUX_TYPES[0], SELECTED_FLUX_TYPES[1]
    flux_a, _ = photometry[ft_a]
    flux_b, _ = photometry[ft_b]
    mag_a = flux_to_ab_mag(flux_a[:, 2])
    mag_b = flux_to_ab_mag(flux_b[:, 2])
    dmag = mag_b - mag_a
    valid = np.isfinite(mag_a) & np.isfinite(dmag)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(mag_a[valid], dmag[valid], s=2, alpha=0.2, color="steelblue")
    h = ax.hist2d(
        mag_a[valid], dmag[valid], bins=[120, 100], cmin=2, cmap="viridis",
        range=[[18, 28], [-1, 1]], norm=LogNorm(),
    )
    plt.colorbar(h[3], ax=ax, label="Count")
    ax.axhline(0, color="r", ls="--", lw=2)
    ax.set_xlabel(f"{ft_a} i-mag", fontsize=15)
    ax.set_ylabel(f"{ft_b} - {ft_a} i-mag", fontsize=15)
    ax.set_title(f"Magnitude-Dependent Residual: {ft_b} - {ft_a}")
    ax.set_xlim(18, 28); ax.set_ylim(-1, 1)
    ax.grid(linestyle="--", linewidth=2, alpha=0.6)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"mag_residual_{ft_a}_vs_{ft_b}.png",
                dpi=DPI, bbox_inches="tight")
    plt.show()

## 7. Quality Cuts

In [ ]:
def apply_quality_cuts(flux, flux_err, redshift, object_type,
                       snr_threshold=SNR_THRESHOLD,
                       z_min=Z_MIN, z_max=Z_MAX,
                       allowed_types=ALLOWED_OBJECT_TYPES):
    """Apply sequential quality cuts with attrition tracking."""
    n_total = len(flux)
    mask = np.ones(n_total, dtype=bool)
    cuts_log = [("Initial", n_total, 0)]

    cut = np.isfinite(flux).all(axis=1) & np.isfinite(flux_err).all(axis=1)
    removed = mask.sum() - (mask & cut).sum()
    mask &= cut
    cuts_log.append(("Finite flux (all bands)", mask.sum(), removed))

    cut = (flux > 0).all(axis=1)
    removed = mask.sum() - (mask & cut).sum()
    mask &= cut
    cuts_log.append(("Positive flux (all bands)", mask.sum(), removed))

    i_snr = compute_snr(flux[:, 2], flux_err[:, 2])
    cut = np.isfinite(i_snr) & (i_snr >= snr_threshold)
    removed = mask.sum() - (mask & cut).sum()
    mask &= cut
    cuts_log.append((f"i-band SNR >= {snr_threshold}", mask.sum(), removed))

    cut = (redshift >= z_min) & (redshift <= z_max) & np.isfinite(redshift)
    removed = mask.sum() - (mask & cut).sum()
    mask &= cut
    cuts_log.append((f"Redshift in [{z_min}, {z_max}]", mask.sum(), removed))

    cut = np.array([t in allowed_types for t in object_type])
    removed = mask.sum() - (mask & cut).sum()
    mask &= cut
    cuts_log.append((f"Object type filter", mask.sum(), removed))

    return mask, cuts_log

In [ ]:
quality_masks = {}

for ft in SELECTED_FLUX_TYPES:
    flux, flux_err = photometry[ft]
    mask, cuts_log = apply_quality_cuts(flux, flux_err, redshift, object_type)
    quality_masks[ft] = (mask, cuts_log)

    print(f"\n=== Quality cuts: {ft} ===")
    print(f"{'Cut':<35} {'Remaining':>10} {'Removed':>10} {'Survival':>10}")
    print("-" * 70)
    for name, remaining, removed in cuts_log:
        pct = remaining / len(flux) * 100
        print(f"{name:<35} {remaining:>10,} {removed:>10,} {pct:>9.1f}%")

In [ ]:
# --- Figures 26-28: Attrition waterfall per flux type ---
for ft in SELECTED_FLUX_TYPES:
    mask, cuts_log = quality_masks[ft]

    fig, ax = plt.subplots(figsize=(10, 5))
    names = [c[0] for c in cuts_log]
    remaining = [c[1] for c in cuts_log]

    bars = ax.bar(range(len(names)), remaining, color="steelblue", edgecolor="white")
    for i in range(1, len(names)):
        removed = cuts_log[i][2]
        if removed > 0:
            ax.bar(i, removed, bottom=remaining[i], color="salmon",
                   edgecolor="white", alpha=0.7)

    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Objects remaining")
    ax.set_title(f"Quality Cut Attrition: {ft}")
    for i, (bar, rem) in enumerate(zip(bars, remaining)):
        pct = rem / cuts_log[0][1] * 100
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{rem:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"attrition_waterfall_{ft}.png", dpi=DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# --- Figure 29: Pre/post-cut redshift distribution ---
fig, axes = plt.subplots(1, len(SELECTED_FLUX_TYPES),
                         figsize=(7 * len(SELECTED_FLUX_TYPES), 5))
if len(SELECTED_FLUX_TYPES) == 1:
    axes = [axes]

z_bins = np.arange(0, 5, 0.05)
for ax, ft in zip(axes, SELECTED_FLUX_TYPES):
    mask, _ = quality_masks[ft]
    ax.hist(redshift, bins=z_bins, alpha=0.5, color="gray",
            label=f"Before cuts (N={len(redshift):,})", density=True)
    ax.hist(redshift[mask], bins=z_bins, alpha=0.7, color="steelblue",
            label=f"After cuts (N={mask.sum():,})", density=True)
    ax.set_xlabel("Redshift")
    ax.set_ylabel("Density")
    ax.set_title(f"Redshift Distribution: {ft}")
    ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "redshift_pre_post_cuts.png", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 30: Post-cut mag-z confirmation ---
ft = SELECTED_FLUX_TYPES[0]
mask, _ = quality_masks[ft]
flux_cut, _ = photometry[ft]
i_mag_cut = flux_to_ab_mag(flux_cut[:, 2])
phot_label = PHOT_LABELS.get(ft, ft)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

valid_pre = np.isfinite(i_mag_cut)
plot_mag_z(
    i_mag_cut[valid_pre], redshift[valid_pre], ax=ax1,
    phot=phot_label, label=r"$\rm Pre$-$\rm cut$", cmin=3, add_colorbar=False,
)

valid_post = mask & np.isfinite(i_mag_cut)
plot_mag_z(
    i_mag_cut[valid_post], redshift[valid_post], ax=ax2,
    phot=phot_label, label=r"$\rm Post$-$\rm cut$", cmin=3, add_colorbar=False,
)

fig.suptitle(f"Quality Cut Effect: {ft} i-mag vs z", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"post_cut_mag_z_{ft}.png", dpi=DPI, bbox_inches="tight")
plt.show()

## 8. Extinction Correction + KDE Bandwidth

In [ ]:
export_data = {}

for ft in SELECTED_FLUX_TYPES:
    mask, cuts_log = quality_masks[ft]
    flux_raw, flux_err_raw = photometry[ft]

    flux_cut = flux_raw[mask]
    flux_err_cut = flux_err_raw[mask]
    ext_cut = extinction[mask]
    z_cut = redshift[mask]
    oid_cut = np.array(cat["object_id"])[mask]

    flux_corrected = apply_extinction_correction(flux_cut, ext_cut)
    flux_err_final = flux_err_cut.copy()

    kde_bandwidth = np.maximum(
        KDE_BANDWIDTH_FRACTION * z_cut, KDE_BANDWIDTH_FLOOR
    )

    mask_arr = np.ones_like(flux_corrected, dtype=int)

    export_data[ft] = {
        "flux": flux_corrected,
        "flux_err": flux_err_final,
        "mask": mask_arr,
        "redshifts": z_cut,
        "redshift_errs": kde_bandwidth,
        "object_ids": oid_cut,
    }

    print(f"\n{ft}: {mask.sum():,} objects after cuts")
    print(f"  Extinction correction: median A_i = {np.median(ext_cut[:, 2]):.4f} mag")
    print(f"  KDE bandwidth: median = {np.median(kde_bandwidth):.4f}, "
          f"range = [{kde_bandwidth.min():.4f}, {kde_bandwidth.max():.4f}]")
    print(f"  Corrected i-mag: median = {np.median(flux_to_ab_mag(flux_corrected[:, 2])):.2f}")

## 9. Export

In [ ]:
for ft in SELECTED_FLUX_TYPES:
    d = export_data[ft]

    photo_data = PhotoData(
        flux=d["flux"].astype(np.float64),
        flux_err=d["flux_err"].astype(np.float64),
        mask=d["mask"],
        redshifts=d["redshifts"].astype(np.float64),
        redshift_errs=d["redshift_errs"].astype(np.float64),
        object_ids=d["object_ids"],
        band_names=BANDS,
    )
    photo_data.validate()

    out_dir = OUTPUT_DIR / ft
    out_dir.mkdir(parents=True, exist_ok=True)
    hdf5_path = out_dir / "train.hdf5"
    write_hdf5(photo_data, hdf5_path)
    print(f"  Wrote {hdf5_path} ({photo_data.n_objects:,} objects, {photo_data.n_bands} bands)")

    _, cuts_log = quality_masks[ft]
    metadata = {
        "source_catalog": str(CATALOG_PATH),
        "flux_type": ft,
        "flux_pattern": FLUX_TYPE_REGISTRY[ft][0],
        "fluxerr_pattern": FLUX_TYPE_REGISTRY[ft][1],
        "bands": BANDS,
        "ab_zeropoint": AB_ZP,
        "flux_units": "nJy (extinction-corrected)",
        "n_objects": int(photo_data.n_objects),
        "n_bands": int(photo_data.n_bands),
        "quality_cuts": [
            {"name": name, "remaining": int(rem), "removed": int(rmv)}
            for name, rem, rmv in cuts_log
        ],
        "extinction_correction": "mag subtraction (A_b), errors untouched",
        "kde_bandwidth": {
            "method": "max(fraction * z_spec, floor)",
            "fraction": KDE_BANDWIDTH_FRACTION,
            "floor": KDE_BANDWIDTH_FLOOR,
        },
        "redshift_errs_contains": "KDE bandwidth (NOT raw redshift errors)",
    }
    meta_path = out_dir / "metadata.yaml"
    with open(meta_path, "w") as f:
        yaml.dump(metadata, f, default_flow_style=False, sort_keys=False)
    print(f"  Wrote {meta_path}")

In [ ]:
# Round-trip validation
print("Round-trip validation:")
for ft in SELECTED_FLUX_TYPES:
    hdf5_path = OUTPUT_DIR / ft / "train.hdf5"
    loaded = read_hdf5(hdf5_path)
    loaded.validate()

    original = export_data[ft]
    assert loaded.n_objects == len(original["redshifts"]), "Object count mismatch"
    assert loaded.n_bands == N_BANDS, "Band count mismatch"
    assert np.allclose(loaded.flux, original["flux"], atol=1e-10), "Flux mismatch"
    assert np.allclose(loaded.flux_err, original["flux_err"], atol=1e-10), "Flux err mismatch"
    assert np.allclose(loaded.redshifts, original["redshifts"], atol=1e-10), "Redshift mismatch"
    assert np.allclose(loaded.redshift_errs, original["redshift_errs"], atol=1e-10), "KDE bw mismatch"
    assert loaded.band_names == BANDS, "Band names mismatch"

    print(f"  {ft}: PASS ({loaded.n_objects:,} objects, {loaded.n_bands} bands)")

print("\nAll round-trip validations passed.")

## 10. Summary

In [ ]:
print("\n" + "=" * 70)
print("EXPORT SUMMARY")
print("=" * 70)
print(f"{'Flux Type':<15} {'Objects':>10} {'Survival':>10} {'HDF5 Path'}")
print("-" * 70)
for ft in SELECTED_FLUX_TYPES:
    n_out = export_data[ft]["flux"].shape[0]
    survival = n_out / len(cat) * 100
    path = OUTPUT_DIR / ft / "train.hdf5"
    print(f"{ft:<15} {n_out:>10,} {survival:>9.1f}% {path}")

print(f"\nFigures saved to: {FIGURE_DIR}/")
print(f"Total figures: {len(list(FIGURE_DIR.glob('*.png')))}")
print("\nDone!")